# 07 — Kronos Foundation Model Zero-Shot Forecasts
**GeoSentinel Terminal (VARTA) · Team 7 Lambda · SP2026**

Runs zero-shot price and volatility forecasts for all 12 assets using the
**Kronos-base** foundation model (published at the Association for the Advancement
of Artificial Intelligence (AAAI) 2026, Amazon and Tsinghua University collaboration,
MIT license).

- Trained on 12 billion candlestick records from 45 exchanges
- 102.3 million parameters
- Runs on Apple Metal Performance Shaders (MPS) backend — no GPU required
- Context window: 512 tokens max

**Fallback:** If Kronos is unavailable, Extreme Gradient Boosting (XGBoost) predictions
from notebook 05 are used instead.

Outputs: `data/processed/kronos_forecasts.parquet`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [2]:
# ── Install Kronos if not already installed ────────────────────────────────────
# Run this cell once only
# !pip install git+https://github.com/amazon-science/chronos-forecasting.git

In [3]:
import numpy as np
import polars as pl
from datetime import timedelta
from config import DATA_PROC, TICKERS, ASSETS, KRONOS_MODEL_ID, KRONOS_CONTEXT_LEN, KRONOS_DEVICE
from src.utils import log, save_parquet

# ── Load Kronos pipeline ───────────────────────────────────────────────────
kronos_available = False
try:
    import torch
    from chronos import ChronosPipeline
    device = KRONOS_DEVICE if KRONOS_DEVICE == "mps" and torch.backends.mps.is_available() else "cpu"
    pipeline = ChronosPipeline.from_pretrained(
        KRONOS_MODEL_ID,
        device_map=device,
        torch_dtype=torch.float32,
    )
    kronos_available = True
    log.info(f"Kronos-base loaded on {device}")
except Exception as e:
    log.warning(f"Kronos unavailable: {e}")
    log.warning("XGBoost predictions from notebook 05 will be used as fallback in Streamlit tabs")


`torch_dtype` is deprecated! Use `dtype` instead!


2026-04-16 15:38:01 [INFO] varta — Kronos-base loaded on mps


In [4]:
# ── Load prices ────────────────────────────────────────────────────────────────
prices = pl.read_parquet(DATA_PROC / "prices.parquet")
log.info(f"Prices loaded: {prices.shape}")

2026-04-16 15:38:01 [INFO] varta — Prices loaded: (43474, 8)


In [5]:
# ── Run zero-shot forecasts for all 12 tickers ────────────────────────────
# num_samples=20: reduced from 100 — 80% CI remains valid, avoids MPS memory
# pressure that caused >2 GB disk writes and crashed the system (2026-04-16)
HORIZON = 21  # 21 trading days = approximately 1 month forward
NUM_SAMPLES = 20
results = []


def _next_business_days(last_date, n: int) -> list:
    """
    Return n business dates (Mon–Fri) starting the day after last_date.

    Args:
        last_date: datetime.date — the last known price date.
        n: number of forward business days to generate.
    Returns:
        List of datetime.date objects.
    """
    dates = []
    current = last_date
    while len(dates) < n:
        current = current + timedelta(days=1)
        if current.weekday() < 5:  # Mon=0 … Fri=4
            dates.append(current)
    return dates


if kronos_available:
    for ticker in TICKERS:
        asset_prices = prices.filter(pl.col("ticker") == ticker).sort("date")
        close_arr = asset_prices["close"].to_numpy()
        last_date  = asset_prices["date"][-1]  # datetime.date from Polars

        ctx_tensor = torch.tensor(
            close_arr[-KRONOS_CONTEXT_LEN:], dtype=torch.float32
        ).unsqueeze(0)

        # predict() → Tensor (batch=1, num_samples, horizon)
        raw = pipeline.predict(
            ctx_tensor,
            prediction_length=HORIZON,
            num_samples=NUM_SAMPLES,
        )
        samples = raw.squeeze(0).numpy()  # shape: (NUM_SAMPLES, HORIZON)

        # Release MPS buffers immediately — prevents Metal memory pressure
        # accumulating across tickers (root cause of 2026-04-16 crash)
        del ctx_tensor, raw
        if device == "mps":
            torch.mps.empty_cache()

        mean_fc = samples.mean(axis=0)
        low_80  = np.percentile(samples, 10, axis=0)
        high_80 = np.percentile(samples, 90, axis=0)

        # 3-point rolling smooth on CI bands — removes percentile sampling jitter
        # from low num_samples. Edge-replication padding prevents the zero-pad
        # artifact that mode='same' introduces at day 1 and day 21.
        def _smooth3(arr: np.ndarray) -> np.ndarray:
            padded = np.concatenate([[arr[0]], arr, [arr[-1]]])
            return np.convolve(padded, np.ones(3) / 3, mode="valid")
        low_80  = _smooth3(low_80)
        high_80 = _smooth3(high_80)

        forecast_dates = _next_business_days(last_date, HORIZON)

        for i in range(HORIZON):
            results.append({
                "ticker":        ticker,
                "forecast_date": forecast_dates[i],
                "mean":          float(mean_fc[i]),
                "low_80":        float(low_80[i]),
                "high_80":       float(high_80[i]),
                "last_close":    float(close_arr[-1]),
            })
        log.info(f"  {ticker}: forecast complete ({HORIZON} days, {NUM_SAMPLES} samples)")
else:
    log.warning("Skipping Kronos — creating empty placeholder DataFrame")
    results = [{"ticker": t, "forecast_date": None, "mean": None,
                "low_80": None, "high_80": None, "last_close": None} for t in TICKERS]

forecasts_df = pl.DataFrame(results)
print(f"Forecasts shape: {forecasts_df.shape}")
forecasts_df.head()


2026-04-16 15:38:02 [INFO] varta —   REMX: forecast complete (21 days, 20 samples)


2026-04-16 15:38:03 [INFO] varta —   LIT: forecast complete (21 days, 20 samples)


2026-04-16 15:38:04 [INFO] varta —   ALB: forecast complete (21 days, 20 samples)


2026-04-16 15:38:04 [INFO] varta —   FCX: forecast complete (21 days, 20 samples)


2026-04-16 15:38:05 [INFO] varta —   NVDA: forecast complete (21 days, 20 samples)


...

2026-04-16 15:38:08 [INFO] varta —   CVX: forecast complete (21 days, 20 samples)


2026-04-16 15:38:08 [INFO] varta —   GLD: forecast complete (21 days, 20 samples)


2026-04-16 15:38:09 [INFO] varta —   SPY: forecast complete (21 days, 20 samples)


Forecasts shape: (252, 6)


ticker,forecast_date,mean,low_80,high_80,last_close
str,datetime[μs],f64,f64,f64,f64
"""REMX""",2024-12-31 00:00:00,38.791985,37.93379,39.823265,38.654472
"""REMX""",2025-01-01 00:00:00,38.791988,37.789553,39.981923,38.654472
"""REMX""",2025-01-02 00:00:00,39.029976,37.630894,40.414628,38.654472
"""REMX""",2025-01-03 00:00:00,39.311226,37.342424,40.83291,38.654472
"""REMX""",2025-01-06 00:00:00,39.138153,37.169342,41.150227,38.654472


In [6]:
# ── Save ──────────────────────────────────────────────────────────────────────
save_parquet(forecasts_df, DATA_PROC / "kronos_forecasts.parquet", "Kronos forecasts")
print("Saved → data/processed/kronos_forecasts.parquet")

2026-04-16 15:38:09 [INFO] varta — Saved Kronos forecasts → /Users/taruntheegela/Desktop/VARTA/data/processed/kronos_forecasts.parquet (252 rows)


Saved → data/processed/kronos_forecasts.parquet


In [7]:
# ── Fan chart preview for one ticker ──────────────────────────────────────
import plotly.graph_objects as go
from src.utils import set_dark_theme

if kronos_available:
    ticker = "NVDA"
    hist = prices.filter(pl.col("ticker") == ticker).sort("date").tail(120)
    fc   = forecasts_df.filter(pl.col("ticker") == ticker)

    fc_dates  = fc["forecast_date"].to_list()
    fc_mean   = fc["mean"].to_list()
    fc_low    = fc["low_80"].to_list()
    fc_high   = fc["high_80"].to_list()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=hist["date"].to_list(), y=hist["close"].to_list(),
        name="Historical", line=dict(color="#94A3B8"),
    ))
    fig.add_trace(go.Scatter(
        x=fc_dates, y=fc_mean,
        name="Kronos Mean Forecast", line=dict(color="#FACC15", dash="dash"),
    ))
    fig.add_trace(go.Scatter(
        x=fc_dates + fc_dates[::-1],
        y=fc_high + fc_low[::-1],
        fill="toself", fillcolor="rgba(250,204,21,0.15)",
        line=dict(color="rgba(0,0,0,0)"), name="80% Confidence Interval",
    ))
    fig.update_layout(
        title=f"{ticker} ({ASSETS[ticker]['name']}) — Kronos 21-Day Zero-Shot Forecast"
    )
    fig = set_dark_theme(fig)
    fig.show()
else:
    print("Kronos not available — fan chart skipped")
